In [1]:
from __future__ import annotations
from dataclasses import dataclass
from collections import defaultdict
import math
import pulp
from routing import Node, Edge, Group, Flow, Topology, solve, print_solution
from viz import show_topology, show_routing

In [2]:
gpu0 = Node("GPU0", capacity=80)
gpu1 = Node("GPU1", capacity=80)
gpu2 = Node("GPU2", capacity=80)
gpu3 = Node("GPU3", capacity=80)
nic0 = Node("NIC0")
nic1 = Node("NIC1")

# groups
nv01 = Group("nv01", 300.0)
nv01r = Group("nv01r", 300.0)
nv23 = Group("nv23", 300.0)
nv23r = Group("nv23r", 300.0)
pxb = Group("pxb", 31.5)
pix = Group("pix", 31.5)
sys_link = Group("sys", 20.0)  # guess, adjust later

topo = Topology(
    nodes=[gpu0, gpu1, gpu2, gpu3, nic0, nic1],
    edges=[
        # NVLink pairs (independent, full duplex)
        Edge(gpu0, gpu1, nv01),
        Edge(gpu1, gpu0, nv01r),
        Edge(gpu2, gpu3, nv23),
        Edge(gpu3, gpu2, nv23r),
        # PXB pairs (all shared)
        Edge(gpu0, gpu2, pxb),
        Edge(gpu2, gpu0, pxb),
        Edge(gpu0, gpu3, pxb),
        Edge(gpu3, gpu0, pxb),
        Edge(gpu1, gpu2, pxb),
        Edge(gpu2, gpu1, pxb),
        Edge(gpu1, gpu3, pxb),
        Edge(gpu3, gpu1, pxb),
        # PIX
        Edge(gpu0, nic0, pix),
        Edge(nic0, gpu0, pix),
        # PXB to NIC0 (GPU1,2,3 to NIC0 are PXB)
        Edge(gpu1, nic0, pxb),
        Edge(nic0, gpu1, pxb),
        Edge(gpu2, nic0, pxb),
        Edge(nic0, gpu2, pxb),
        Edge(gpu3, nic0, pxb),
        Edge(nic0, gpu3, pxb),
        # SYS (everything to NIC1)
        Edge(gpu0, nic1, sys_link),
        Edge(nic1, gpu0, sys_link),
        Edge(gpu1, nic1, sys_link),
        Edge(nic1, gpu1, sys_link),
        Edge(gpu2, nic1, sys_link),
        Edge(nic1, gpu2, sys_link),
        Edge(gpu3, nic1, sys_link),
        Edge(nic1, gpu3, sys_link),
        Edge(nic0, nic1, sys_link),
        Edge(nic1, nic0, sys_link),
    ],
)

In [3]:
show_topology(topo, "h100_topo_viz.html")

'h100_topo_viz.html'

In [4]:
gpus = [gpu0, gpu1, gpu2, gpu3]
n = len(gpus)

ring_flows = [
    Flow(gpus[i], gpus[(i + 1) % n], size=40)  # 1 GB each
    for i in range(n)
]

s = solve(topo, ring_flows)
print_solution(s)


Makespan: 1.553398

  GPU0 -> GPU1 (40 bytes):
    GPU0 -> GPU1: 40.00 bytes (100.0%)

  GPU1 -> GPU2 (40 bytes):
    GPU1 -> NIC0 -> GPU2: 40.00 bytes (100.0%)

  GPU2 -> GPU3 (40 bytes):
    GPU2 -> GPU3: 40.00 bytes (100.0%)

  GPU3 -> GPU0 (40 bytes):
    GPU3 -> GPU0: 8.93 bytes (22.3%)
    GPU3 -> NIC1 -> GPU0: 31.07 bytes (77.7%)


In [5]:
show_routing(topo,s, "h100_ring_viz.html")

'h100_ring_viz.html'